# GADS Data Processing
Load GADS CSV files from Lakehouse Files, clean and transform, then save as Delta tables.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

BASE_PATH = "Files/GADS"

## 1. Unit Configuration (dimension table)

In [ ]:
df_units = (spark.read.option("header", True).option("inferSchema", True)
    .csv(f"{BASE_PATH}/T_UNIT_CONFIGURATION.csv")
    .withColumn("COMMISSION_DT", F.to_timestamp("COMMISSION_DT"))
    .withColumn("EFF_END_DT", F.to_timestamp("EFF_END_DT"))
    .withColumn("FAIL_START_DT", F.to_timestamp("FAIL_START_DT"))
    .withColumn("UPDT_DT", F.to_timestamp("UPDT_DT"))
)

df_units.write.mode("overwrite").format("delta").saveAsTable("gads_unit_configuration")
print(f"gads_unit_configuration: {df_units.count()} rows")
df_units.show(5, truncate=False)

## 2. Event Type (lookup)

In [ ]:
df_event_type = (spark.read.option("header", True).option("inferSchema", True)
    .csv(f"{BASE_PATH}/T_EVENT_TYPE.csv")
)

df_event_type.write.mode("overwrite").format("delta").saveAsTable("gads_event_type")
print(f"gads_event_type: {df_event_type.count()} rows")
df_event_type.show(25, truncate=False)

## 3. Event Cause Code (lookup)

In [ ]:
df_cause = (spark.read.option("header", True).option("inferSchema", True)
    .csv(f"{BASE_PATH}/T_EVENT_CAUSE_CODE.csv")
    .withColumn("VALID_FROM_DT", F.to_timestamp("VALID_FROM_DT"))
    .withColumn("VALID_TO_DT", F.to_timestamp("VALID_TO_DT"))
)

df_cause.write.mode("overwrite").format("delta").saveAsTable("gads_event_cause_code")
print(f"gads_event_cause_code: {df_cause.count()} rows")

## 4. Unit Events (fact table)

In [ ]:
df_events = (spark.read.option("header", True).option("inferSchema", True)
    .csv(f"{BASE_PATH}/T_UNIT_EVENT.csv")
    .withColumn("REAL_START_DT", F.to_timestamp("REAL_START_DT"))
    .withColumn("REAL_END_DT", F.to_timestamp("REAL_END_DT"))
    .withColumn("UPDT_DT", F.to_timestamp("UPDT_DT"))
    .withColumn("DURATION_HRS", 
        (F.unix_timestamp("REAL_END_DT") - F.unix_timestamp("REAL_START_DT")) / 3600
    )
    .withColumn("EVENT_YEAR", F.year("REAL_START_DT"))
    .withColumn("EVENT_MONTH", F.month("REAL_START_DT"))
)

df_events.write.mode("overwrite").format("delta").saveAsTable("gads_unit_event")
print(f"gads_unit_event: {df_events.count()} rows")
df_events.select("GEN_SEQ_NO","UNIT_ID","REAL_START_DT","REAL_END_DT","EVENT_TYPE_CD","DURATION_HRS","DERATING","CAUSE_OF_EVENT").show(5, truncate=60)

## 5. Remaining lookup tables

In [ ]:
lookup_tables = [
    "T_EVENT_CONTRIBUTION_CODE",
    "T_EQUIP_TYPE_OUTAGE",
    "T_EQUIP_TYPE",
    "T_FAILURE_ANALYSIS_EVENT",
    "T_FAILURE_TYPE",
    "T_FAILURE_MODE",
    "T_FAILURE_MECHANISM",
    "T_FAILURE_LOCATION",
    "T_CORRECTIVE_ACTION",
    "T_CONTRIBUTION_FACTOR",
    "T_AVAILABILITY_STATISTIC",
    "T_PENDING_EVENT",
    "T_ORGANIZATION",
    "T_GENERATION_TYPE",
    "T_UNIT_TYPE",
    "T_FUEL_ID",
]

for table in lookup_tables:
    delta_name = f"gads_{table[2:].lower()}"
    df = (spark.read.option("header", True).option("inferSchema", True)
        .csv(f"{BASE_PATH}/{table}.csv"))
    df.write.mode("overwrite").format("delta").saveAsTable(delta_name)
    print(f"{delta_name}: {df.count()} rows")

## 6. Event Log (audit trail)

In [ ]:
df_log = (spark.read.option("header", True).option("inferSchema", True)
    .option("multiLine", True).option("escape", '"')
    .csv(f"{BASE_PATH}/T_EVENT_LOG.csv")
    .withColumn("TRANSACTION_DT", F.to_timestamp("TRANSACTION_DT"))
)

df_log.write.mode("overwrite").format("delta").saveAsTable("gads_event_log")
print(f"gads_event_log: {df_log.count()} rows")

## 7. Enriched events view — join events with lookups

In [ ]:
df_enriched = (spark.table("gads_unit_event").alias("e")
    .join(spark.table("gads_event_type").alias("et"),
          F.trim(F.col("e.EVENT_TYPE_CD")) == F.col("et.CODE"), "left")
    .join(spark.table("gads_event_cause_code").alias("cc"),
          F.col("e.EVENT_CAUSE_CD") == F.col("cc.CODE"), "left")
    .join(spark.table("gads_unit_configuration").alias("u"),
          F.col("e.UNIT_ID") == F.col("u.UNIT_ID"), "left")
    .select(
        F.col("e.GEN_SEQ_NO"),
        F.col("e.UNIT_ID"),
        F.col("u.OTS_UNIT_ID").alias("UNIT_NAME"),
        F.col("u.MAX_DESIGN_CAP"),
        F.col("e.REAL_START_DT"),
        F.col("e.REAL_END_DT"),
        F.col("e.DURATION_HRS"),
        F.col("e.EVENT_YEAR"),
        F.col("e.EVENT_MONTH"),
        F.col("e.EVENT_TYPE_CD"),
        F.col("et.DESCR").alias("EVENT_TYPE_DESC"),
        F.col("e.EVENT_CAUSE_CD"),
        F.col("cc.DESCR").alias("CAUSE_CODE_DESC"),
        F.col("e.AVAIL_CAP_DURING"),
        F.col("e.AVAIL_CAP_AFTER"),
        F.col("e.DERATING"),
        F.col("e.EQUIPMENT_DESC"),
        F.col("e.CAUSE_OF_EVENT"),
        F.col("u.PRIMARY_FUEL"),
        F.col("u.UNIT_TYPE"),
        F.col("u.BOILER_MANF"),
        F.col("u.TURBINE_MANF"),
    )
)

df_enriched.write.mode("overwrite").format("delta").saveAsTable("gads_events_enriched")
print(f"gads_events_enriched: {df_enriched.count()} rows")
df_enriched.show(10, truncate=40)

## 8. Quick summary stats

In [ ]:
print("Events by type:")
spark.sql("""
    SELECT EVENT_TYPE_CD, EVENT_TYPE_DESC, COUNT(*) as event_count,
           ROUND(AVG(DURATION_HRS), 1) as avg_duration_hrs,
           ROUND(AVG(DERATING), 1) as avg_derating_mw
    FROM gads_events_enriched
    GROUP BY EVENT_TYPE_CD, EVENT_TYPE_DESC
    ORDER BY event_count DESC
""").show(25, truncate=False)

print("\nTop 10 units by unplanned outage count:")
spark.sql("""
    SELECT UNIT_ID, UNIT_NAME, COUNT(*) as outage_count,
           ROUND(SUM(DURATION_HRS), 0) as total_hours_down,
           ROUND(AVG(DURATION_HRS), 1) as avg_hours_per_event
    FROM gads_events_enriched
    WHERE EVENT_TYPE_CD IN ('U1','U2','U3','SF')
    GROUP BY UNIT_ID, UNIT_NAME
    ORDER BY outage_count DESC
    LIMIT 10
""").show(truncate=False)

print("\nTop 10 cause codes for unplanned outages:")
spark.sql("""
    SELECT EVENT_CAUSE_CD, CAUSE_CODE_DESC, COUNT(*) as count,
           ROUND(AVG(DURATION_HRS), 1) as avg_duration_hrs
    FROM gads_events_enriched
    WHERE EVENT_TYPE_CD IN ('U1','U2','U3')
    GROUP BY EVENT_CAUSE_CD, CAUSE_CODE_DESC
    ORDER BY count DESC
    LIMIT 10
""").show(truncate=False)

In [ ]:
df = spark.sql("SELECT * FROM lh_poc.dbo.gads_events_enriched LIMIT 1000")
display(df)